<span class="ͼ12">
<center><h3>Community Fire Behavior Model (ESMX)</h3></center>
</span>

<button data-commandLinker-command="toc:show-panel">Show ToC</button>

The [Community Fire Behavior Model](https://ral.ucar.edu/model/community-fire-behavior-model) is an [open source](https://github.com/NCAR/fire_behavior) Earth system model developed at NCAR's Research Applications Laboratory. The model has been wrapped in a NUOPC cap and is regularly tested with ESMX.

<span class="ͼ10">
<h2>Prerequisites</h2>
</span>

- Fortran: A Fortran compiler such as gfortran is required.
- MPI: An MPI library is required for `DM_PARALLEL`, which is the default build mode for the CFBM.
- NetCDF: Both ESMF and CFBM utilize NetCDF for file i/o. The library must be installed and the NETCDF_ROOT environment variable should be set to the location of the NetCDF installation.
- ESMF: This tutorial requires an ESMF installation. After installing ESMF the `ESMFMKFILE` environment variable should be set to the location of the `esmf.mk` file wihtin the ESMF installation. The ESMF installation `bin` folder must be included in your current `PATH` in order execute `ESMX_Builder`.
- git: git is 

<span class="ͼ10">
<h2>Delete Previous Artifacts</h2>
</span>

<div class="alert alert-success">
Deleting previous build and run artifacts is recommended but not required. This can be done by deleting the following files from within the current directory.
</div>

<button data-commandLinker-command="docmanager:show-in-file-browser">Open Notebook Directory</button>

- CFBM
- esmxBuildCFBM.yaml
- build
- install
- run

<div class="alert alert-warning">
The following shell commands can also be used to forcefully delete previous build and run artifacts.
</div>

```sh
rm -rf CFBM
rm -f  esmxBuildCFBM.yaml
rm -rf build
rm -rf install
rm -rf run
```

<span class="ͼ10">
<h2>Build CFBM using ESMX</h2>
</span>

<span class="ͼ14">
<h3>Download the CFBM</h3>
</span>

This command will download the CFBM source code into a directory named CFBM.

In [ ]:
!git clone -b develop https://github.com/NCAR/fire_behavior.git CFBM

<span class="ͼ14">
<h3>Build Executable</h3>
</span>

Create an ESMX build file including the CFBM component. Then utilize the `ESMX_Builder` script to build the `esmx_fire` executable. By default, the executable will be installed into `./install/bin`. Make sure to build the CFBM NUOPC cap using the cmake build argument `NUOPC=ON`. CFBM is currently not working as a cmake subdirectory but the `cmake.external` build and link option is working. The CFBM component does not have to share a name with the source directory; in this tutorial we are naming it `fire_behavior`.

In [ ]:
%%writefile esmxBuildCFBM.yaml
application:
  exe_name: esmx_fire
  
components:
  fire_behavior:
    source_dir: CFBM
    build_type: cmake.external
    build_args: -DNUOPC=ON
    libraries: fire_behavior_nuopc firelib
    fort_module: fire_behavior_nuopc.mod

In [ ]:
!ESMX_Builder esmxBuildCFBM.yaml

<span class="ͼ10">
<h2>Test CFBM using ESMX</h2>
</span>

<span class="ͼ14">
<h3>Copy Test Input</h3>
</span>

Copy input files from the CFBM tests directory into a run directory. You will need a CFBM namelist file, CFBM grid file, ESMX field dictionary file, and executable.

In [ ]:
!mkdir -p run
!cp CFBM/tests/testx/geo_em.d01.nc run/.
!cp CFBM/tests/testx/namelist.fire run/.
!cp CFBM/tests/fd_fire.yaml run/.
!cp install/bin/esmx_fire run/.

<span class="ͼ14">
<h3>Create Run Configuration File</h3>
</span>

Create an ESMX run configuration file that includes coupling the ESMX_Data component to the fire_behavior component. Use the runSequence to designate data exchanges between system components.

In [ ]:
%%writefile run/esmxRunCFBM.yaml
ESMX:
  App:
    logAppendFlag:          false
    fieldDictionary:        ./fd_fire.yaml
    startTime:              2012-06-25T18:00:00
    stopTime:               2012-06-25T18:00:10
  Driver:
    componentList:          [FIRE, ATMD]
    runSequence: |
      @1
        ATMD -> FIRE
        FIRE -> ATMD
        FIRE
        ATMD
      @

FIRE:
  model: fire_behavior
  petList:          [0]
  attributes:
    Verbosity: low

ATMD:
  model: ESMX_Data
  petList:          [0]
  attributes:
    Verbosity: low
  geom: { nx: 207, ny: 197, nz: 44,
          minx: -109.370, miny: 36.279,
          maxx: -101.948, maxy: 41.619
        }
  importFields:
  exportFields:
    inst_zonal_wind_levels:   {dim: 3, val: 1}
    inst_merid_wind_levels:   {dim: 3, val: 3}
    inst_geop_levels:         {dim: 3, val: 15046.8}
    inst_pres_levels:         {dim: 3, val: 85000}
    inst_surface_roughness:   {dim: 2, val: 0.12}
    mean_prec_rate:           {dim: 2, val: 0}
    inst_spec_humid_height2m: {dim: 2, val: 0.005}
    inst_pres_height_surface: {dim: 2, val: 85000}
    inst_temp_height2m:       {dim: 2, val: 310}
    inst_pres_height_lowest_from_phys:        {dim: 2, val: 85000}
    inst_spec_humid_height_lowest_from_phys:  {dim: 2, val: 0.005}
    inst_temp_height_lowest_from_phys:        {dim: 2, val: 310}
    inst_zonal_wind_height10m:                {dim: 2, val: 1}
    inst_merid_wind_height10m:                {dim: 2, val: 3}

<span class="ͼ14">
<h3>Run Simulation</h3>
</span>

Execute the esmx_fire simulation from the run directory.

In [ ]:
!cd run && ./esmx_fire esmxRunCFBM.yaml

<span class="ͼ14">
<h3>Review Output</h3>
</span>

Input and Output are located in the run directory.

In [ ]:
!ls run